In [1]:
pip install seaborn

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from brevitas.export import export_qonnx
import onnx
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import gc
from brevitas.nn import QuantConv2d, QuantLinear, QuantReLU, TruncAvgPool2d
from brevitas.quant import Int32Bias
from configparser import ConfigParser
from torch.nn import Sequential
from brevitas.core.restrict_val import RestrictValueType
from brevitas.quant import Int8ActPerTensorFloat
from brevitas.quant import Int8WeightPerTensorFloat,Int8WeightPerChannelFloat
from brevitas.quant import Uint8ActPerTensorFloat
from tqdm import tqdm
import copy
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from brevitas.export import export_qonnx
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from finn.util.visualization import showSrc
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames,RemoveStaticGraphInputs
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes
from qonnx.transformation.fold_constants import FoldConstants
import finn.transformation.streamline.absorb as absorb
import finn.transformation.streamline.reorder as reorder
from finn.transformation.streamline import Streamline
from qonnx.transformation.double_to_single_float import DoubleToSingleFloat
from qonnx.transformation.change_datalayout import ChangeDataLayoutQuantAvgPool2d
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from finn.transformation.streamline.collapse_repeated import CollapseRepeatedMul
from qonnx.transformation.remove import RemoveIdentityOps
from finn.transformation.streamline.round_thresholds import RoundAndClipThresholds
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import CreateDataflowPartition
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import shutil
from finn.transformation.streamline.reorder import MoveScalarLinearPastInvariants, MakeMaxPoolNHWC
from qonnx.transformation.insert_topk import InsertTopK
from finn.transformation.streamline.absorb import AbsorbScalarMulAddIntoTopK,AbsorbAddIntoMultiThreshold,AbsorbMulIntoMultiThreshold
from torchvision.models import vgg16, VGG16_Weights
from sklearn.metrics import accuracy_score, classification_report, cohen_kappa_score, confusion_matrix
from tqdm import tqdm
from brevitas import config
import warnings

## Dataset

In [3]:
BASE_PATH = '/home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/DDR-dataset/DR_grading'
TRAIN_PATH = os.path.join(BASE_PATH, 'train')
VALID_PATH = os.path.join(BASE_PATH, 'valid')
TEST_PATH = os.path.join(BASE_PATH, 'test')

In [4]:
def load_labels(file_path):
    labels = pd.read_csv(file_path, sep=' ', header=None, names=['filename', 'label'])
    return labels

train_labels = load_labels(os.path.join(BASE_PATH, 'train.txt'))
valid_labels = load_labels(os.path.join(BASE_PATH, 'valid.txt'))
test_labels = load_labels(os.path.join(BASE_PATH, 'test.txt'))

In [5]:
class CustomDataset(Dataset):
    def __init__(self, labels, dir_path, transform=None):
        self.labels = labels
        self.dir_path = dir_path
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_name = os.path.join(self.dir_path, self.labels.iloc[idx, 0])
        image = Image.open(img_name).convert('RGB')
        label = int(self.labels.iloc[idx, 1])
        if self.transform:
            image = self.transform(image)
        return image, label

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [7]:
BATCH_SIZE = 16  # Tamanho do batch

train_dataset = CustomDataset(train_labels, TRAIN_PATH, transform=train_transform)
valid_dataset = CustomDataset(valid_labels, VALID_PATH, transform=test_transform)
test_dataset = CustomDataset(test_labels, TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Definindo modelo não quantizado

In [8]:
weights = VGG16_Weights.DEFAULT
model = vgg16(weights=weights)

num_ftrs = model.classifier[0].in_features  # 512*7*7
model.classifier = nn.Sequential(
    nn.Linear(num_ftrs, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 6)
)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /tmp/home_dir/.cache/torch/hub/checkpoints/vgg16-397923af.pth


  0%|          | 0.00/528M [00:00<?, ?B/s]

In [9]:
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

## Definindo modelo quantizado

In [10]:
# Copyright (C) 2023, Advanced Micro Devices, Inc. All rights reserved.
# SPDX-License-Identifier: BSD-3-Clause

class CommonIntWeightPerTensorQuant(Int8WeightPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None

class CommonIntWeightPerChannelQuant(CommonIntWeightPerTensorQuant):
    scaling_per_output_channel = True

class CommonIntActQuant(Int8ActPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None
    restrict_scaling_type = RestrictValueType.LOG_FP

class CommonUintActQuant(Uint8ActPerTensorFloat):
    scaling_min_val = 2e-16
    bit_width = None
    restrict_scaling_type = RestrictValueType.LOG_FP

In [11]:
class QuantVGG16(nn.Module):
    def __init__(self, weight_bit_width, act_bit_width, num_classes=6):
        super(QuantVGG16, self).__init__()
        
        self.features = nn.Sequential(
            # Bloco 1
            QuantConv2d(3, 64, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=8,
                        weight_quant=Int8WeightPerChannelFloat),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(64, 64, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 2
            QuantConv2d(64, 128, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(128, 128, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 3
            QuantConv2d(128, 256, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(256, 256, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(256, 256, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 4
            QuantConv2d(256, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Bloco 5
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            
            QuantConv2d(512, 512, kernel_size=3, stride=1, padding=1,groups=1,
                        bias=True, weight_bit_width=weight_bit_width,
                        weight_quant=CommonIntWeightPerChannelQuant),
            QuantReLU(act_quant=CommonUintActQuant,
                      bit_width=act_bit_width, return_quant_tensor=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
        )


        self.classifier = nn.Sequential(
            QuantLinear(512*7*7, 
                        256, 
                        bias=True, 
                        weight_quant=CommonIntWeightPerChannelQuant,
                        weight_bit_width=weight_bit_width),
            QuantReLU(act_quant=CommonUintActQuant, 
                      bit_width=act_bit_width, 
                      return_quant_tensor=True),
            nn.Dropout(0.4),
            QuantLinear(256, 
                        num_classes, 
                        bias=True, 
                        weight_quant=Int8WeightPerTensorFloat,
                        weight_bit_width=8),
        )
        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

## Treinando o modelo não quantizado 

In [24]:
def train_model(model, criterion, optimizer, train_loader, valid_loader, model_name, num_epochs=20):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                data_loader = train_loader
            else:
                model.eval()
                data_loader = valid_loader
                
            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []
            
            for inputs, labels in data_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            epoch_loss = running_loss / len(data_loader.dataset)
            epoch_acc = running_corrects.double() / len(data_loader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(best_model_wts, model_name)
    
    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

In [23]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [25]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)

In [26]:
model = train_model(model, criterion, optimizer, train_loader, valid_loader,"best_vgg16.pth", num_epochs=20)

Epoch 1/20
----------
train Loss: 1.0560 Acc: 0.5927
val Loss: 0.9256 Acc: 0.6400
Epoch 2/20
----------
train Loss: 0.8912 Acc: 0.6732
val Loss: 0.7165 Acc: 0.7541
Epoch 3/20
----------
train Loss: 0.8235 Acc: 0.6998
val Loss: 0.6071 Acc: 0.8156
Epoch 4/20
----------
train Loss: 0.7671 Acc: 0.7204
val Loss: 0.6196 Acc: 0.7914
Epoch 5/20
----------
train Loss: 0.7380 Acc: 0.7323
val Loss: 0.6948 Acc: 0.7658
Epoch 6/20
----------
train Loss: 0.6978 Acc: 0.7484
val Loss: 0.6332 Acc: 0.7797
Epoch 7/20
----------
train Loss: 0.6677 Acc: 0.7573
val Loss: 0.6096 Acc: 0.8138
Epoch 8/20
----------
train Loss: 0.6589 Acc: 0.7642
val Loss: 0.5916 Acc: 0.8123
Epoch 9/20
----------
train Loss: 0.6375 Acc: 0.7719
val Loss: 0.6404 Acc: 0.7849
Epoch 10/20
----------
train Loss: 0.6265 Acc: 0.7775
val Loss: 0.6120 Acc: 0.8072
Epoch 11/20
----------
train Loss: 0.6157 Acc: 0.7789
val Loss: 0.6706 Acc: 0.8086
Epoch 12/20
----------
train Loss: 0.6009 Acc: 0.7893
val Loss: 0.6403 Acc: 0.7914
Epoch 13/20
-

Avaliando o modelo

## Avaliando o Modelo Não Quantizado

In [27]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load("best_vgg16.pth", map_location=device))
model = model.to(device)
model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [13]:
def calculate_metrics(loader, model, device):
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)], output_dict=True)
    kappa = cohen_kappa_score(all_labels, all_preds)
    
    per_class_accuracy = [report[str(i)]['precision'] for i in range(6)]
    mean_accuracy = sum(per_class_accuracy) / len(per_class_accuracy)

    return accuracy, mean_accuracy, kappa, classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)]), confusion_matrix(all_labels, all_preds)

In [14]:
def plot_confusion_matrix(cm, title, filename):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[str(i) for i in range(6)], yticklabels=[str(i) for i in range(6)])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.savefig(filename)
    plt.close()

In [30]:
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report:")
print(valid_report)
print("Validation Confusion Matrix:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_vgg16.png')

Validation Set Metrics:
Validation OA: 0.8240
Validation AA: 0.6868
Validation Kappa: 0.7241
Validation Classification Report:
              precision    recall  f1-score   support

           0       0.81      1.00      0.89      1253
           1       0.33      0.02      0.03       126
           2       0.85      0.75      0.80       895
           3       0.42      0.28      0.33        47
           4       0.72      0.75      0.74       182
           5       0.98      0.81      0.89       230

    accuracy                           0.82      2733
   macro avg       0.69      0.60      0.61      2733
weighted avg       0.81      0.82      0.80      2733

Validation Confusion Matrix:
[[1248    0    5    0    0    0]
 [  94    2   27    0    3    0]
 [ 181    3  667   16   24    4]
 [   0    0   28   13    6    0]
 [   6    1   37    2  136    0]
 [   8    0   17    0   19  186]]


In [32]:
warnings.filterwarnings("ignore", message='UndefinedMetricWarning')
print("Test Set Metrics:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report:")
print(test_report)
print("Test Confusion Matrix:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_vgg16.png')

Test Set Metrics:
Test OA: 0.7111
Test AA: 0.5995
Test Kappa: 0.5371
Test Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.99      0.81      1880
           1       0.00      0.00      0.00       189
           2       0.77      0.41      0.54      1344
           3       0.57      0.18      0.28        71
           4       0.77      0.65      0.70       275
           5       0.81      0.90      0.85       346

    accuracy                           0.71      4105
   macro avg       0.60      0.52      0.53      4105
weighted avg       0.69      0.71      0.67      4105

Test Confusion Matrix:
[[1864    0   16    0    0    0]
 [ 152    0   36    0    1    0]
 [ 707    0  554    7   22   54]
 [   9    0   44   13    5    0]
 [  13    0   63    3  178   18]
 [   4    0    8    0   24  310]]


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

## Treinando o modelos quantizados

In [11]:
model.load_state_dict(torch.load("best_vgg16.pth"))

<All keys matched successfully>

In [12]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [13]:
def train_qat(model, criterion, optimizer, train_loader, valid_loader, model_name, num_epochs=10):
    best_model_wts = model.state_dict()
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)
        
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                data_loader = train_loader
            else:
                model.eval()
                data_loader = valid_loader
                
            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []
            
            for inputs, labels in data_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
            
            epoch_loss = running_loss / len(data_loader.dataset)
            epoch_acc = running_corrects.double() / len(data_loader.dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(best_model_wts, model_name)
    
    print(f'Best val Acc: {best_acc:4f}')
    model.load_state_dict(best_model_wts)
    return model

In [14]:
BATCH_SIZE = 8  # Tamanho do batch reduzido para caber na GPU

train_dataset = CustomDataset(train_labels, TRAIN_PATH, transform=train_transform)
valid_dataset = CustomDataset(valid_labels, VALID_PATH, transform=test_transform)
test_dataset = CustomDataset(test_labels, TEST_PATH, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [21]:
print("QAT para pesos de 8 bits e ativações de 8 bits.")
model_name = 'qat_vgg16_w8_a8.pth'

quant_model = QuantVGG16(8,8)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 8 bits e ativações de 8 bits.
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_187/2399627119.py:118: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 0.5775 Acc: 0.7994
val Loss: 0.6218 Acc: 0.8196
Epoch 2/10
----------
train Loss: 0.5821 Acc: 0.7963
val Loss: 0.5958 Acc: 0.8134
Epoch 3/10
----------
train Loss: 0.5708 Acc: 0.7952
val Loss: 0.5708 Acc: 0.8181
Epoch 4/10
----------
train Loss: 0.5662 Acc: 0.7999
val Loss: 0.6495 Acc: 0.7988
Epoch 5/10
----------
train Loss: 0.5362 Acc: 0.8110
val Loss: 0.6873 Acc: 0.7516
Epoch 6/10
----------
train Loss: 0.5312 Acc: 0.8116
val Loss: 0.5760 Acc: 0.8149
Epoch 7/10
----------
train Loss: 0.5153 Acc: 0.8148
val Loss: 0.5411 Acc: 0.8269
Epoch 8/10
----------
train Loss: 0.5051 Acc: 0.8212
val Loss: 0.6089 Acc: 0.8057
Epoch 9/10
----------
train Loss: 0.5056 Acc: 0.8129
val Loss: 0.5618 Acc: 0.8178
Epoch 10/10
----------
train Loss: 0.5036 Acc: 0.8170
val Loss: 0.6622 Acc: 0.8057
Best val Acc: 0.826930


Treino sem loop dando restart no kernel

In [17]:
print("QAT para pesos de 8 bits e ativações de 4 bits.")
model_name = 'qat_vgg16_w8_a4.pth'

quant_model = QuantVGG16(8,4)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 8 bits e ativações de 4 bits.
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_260/2399627119.py:118: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 0.6545 Acc: 0.7662
val Loss: 0.5922 Acc: 0.8090
Epoch 2/10
----------
train Loss: 0.6392 Acc: 0.7683
val Loss: 0.6509 Acc: 0.7764
Epoch 3/10
----------
train Loss: 0.6240 Acc: 0.7769
val Loss: 0.8338 Acc: 0.7611
Epoch 4/10
----------
train Loss: 0.6188 Acc: 0.7769
val Loss: 0.5542 Acc: 0.8280
Epoch 5/10
----------
train Loss: 0.6151 Acc: 0.7778
val Loss: 0.6527 Acc: 0.7662
Epoch 6/10
----------
train Loss: 0.6248 Acc: 0.7671
val Loss: 0.7064 Acc: 0.7783
Epoch 7/10
----------
train Loss: 0.5944 Acc: 0.7892
val Loss: 0.6750 Acc: 0.7940
Epoch 8/10
----------
train Loss: 0.6061 Acc: 0.7786
val Loss: 0.6023 Acc: 0.8116
Epoch 9/10
----------
train Loss: 0.5870 Acc: 0.7902
val Loss: 0.5348 Acc: 0.8251
Epoch 10/10
----------
train Loss: 0.5838 Acc: 0.7861
val Loss: 0.6931 Acc: 0.7925
Best val Acc: 0.828028


In [15]:
print("QAT para pesos de 4 bits e ativações de 8 bits.")
model_name = 'qat_vgg16_w4_a8.pth'

quant_model = QuantVGG16(4,8)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 4 bits e ativações de 8 bits.
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_193/1034992462.py:117: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 0.5901 Acc: 0.7912
val Loss: 0.6027 Acc: 0.8160
Epoch 2/10
----------
train Loss: 0.5817 Acc: 0.7895
val Loss: 0.6304 Acc: 0.8119
Epoch 3/10
----------
train Loss: 0.5722 Acc: 0.7977
val Loss: 0.6071 Acc: 0.8090
Epoch 4/10
----------
train Loss: 0.5636 Acc: 0.8010
val Loss: 0.5729 Acc: 0.8266
Epoch 5/10
----------
train Loss: 0.5648 Acc: 0.7982
val Loss: 0.7097 Acc: 0.7852
Epoch 6/10
----------
train Loss: 0.5429 Acc: 0.8064
val Loss: 0.5362 Acc: 0.8375
Epoch 7/10
----------
train Loss: 0.5309 Acc: 0.8146
val Loss: 0.6495 Acc: 0.7922
Epoch 8/10
----------
train Loss: 0.5265 Acc: 0.8178
val Loss: 0.6501 Acc: 0.8145
Epoch 9/10
----------
train Loss: 0.5132 Acc: 0.8202
val Loss: 0.6149 Acc: 0.7995
Epoch 10/10
----------
train Loss: 0.5119 Acc: 0.8178
val Loss: 0.6718 Acc: 0.7977
Best val Acc: 0.837541


In [ ]:
print("QAT para pesos de 4 bits e ativações de 4 bits.")
model_name = 'qat_vgg16_w4_a4.pth'

quant_model = QuantVGG16(4,4)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

In [16]:
print("QAT para pesos de 8 bits e ativações de 2 bits.")
model_name = 'qat_vgg16_w8_a2.pth'

quant_model = QuantVGG16(8,2)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 8 bits e ativações de 2 bits.
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_34543/1034992462.py:117: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 1.3759 Acc: 0.4278
val Loss: 1.2662 Acc: 0.4636
Epoch 2/10
----------
train Loss: 1.3052 Acc: 0.4535
val Loss: 1.2495 Acc: 0.4852
Epoch 3/10
----------
train Loss: 1.2928 Acc: 0.4674
val Loss: 1.2454 Acc: 0.4746
Epoch 4/10
----------
train Loss: 1.2871 Acc: 0.4748
val Loss: 1.2516 Acc: 0.4771
Epoch 5/10
----------
train Loss: 1.2839 Acc: 0.4698
val Loss: 1.3117 Acc: 0.4709
Epoch 6/10
----------
train Loss: 1.2797 Acc: 0.4696
val Loss: 1.2590 Acc: 0.4830
Epoch 7/10
----------
train Loss: 1.2674 Acc: 0.4761
val Loss: 1.2195 Acc: 0.5093
Epoch 8/10
----------
train Loss: 1.2633 Acc: 0.4805
val Loss: 1.2385 Acc: 0.4885
Epoch 9/10
----------
train Loss: 1.2672 Acc: 0.4767
val Loss: 1.2440 Acc: 0.5020
Epoch 10/10
----------
train Loss: 1.2594 Acc: 0.4868
val Loss: 1.2643 Acc: 0.4896
Best val Acc: 0.509330


In [15]:
print("QAT para pesos de 2 bits e ativações de 8 bits.")
model_name = 'qat_vgg16_w2_a8.pth'

quant_model = QuantVGG16(2,8)# definindo modelo
quant_model.load_state_dict(model.state_dict(), strict=False) # copiando pesos
quant_model.to(device) #movendo para a gpu

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(quant_model.parameters(),lr=0.0001)

quant_model = train_qat(quant_model,criterion,optimizer,train_loader,valid_loader,model_name)

QAT para pesos de 2 bits e ativações de 8 bits.
Epoch 1/10
----------


/usr/local/lib/python3.10/dist-packages/torch/_tensor.py:1255: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1758.)
  return super(Tensor, self).rename(names)
/tmp/ipykernel_39856/1034992462.py:117: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


train Loss: 1.3028 Acc: 0.4802
val Loss: 1.1105 Acc: 0.5401
Epoch 2/10
----------
train Loss: 1.1508 Acc: 0.5400
val Loss: 1.1224 Acc: 0.5353
Epoch 3/10
----------
train Loss: 1.1029 Acc: 0.5677
val Loss: 1.0332 Acc: 0.5807
Epoch 4/10
----------
train Loss: 1.0706 Acc: 0.5855
val Loss: 1.0097 Acc: 0.5865
Epoch 5/10
----------
train Loss: 1.0491 Acc: 0.5934
val Loss: 0.9998 Acc: 0.6231
Epoch 6/10
----------
train Loss: 1.0169 Acc: 0.6059
val Loss: 0.9699 Acc: 0.6140
Epoch 7/10
----------
train Loss: 1.0030 Acc: 0.6099
val Loss: 0.9743 Acc: 0.6187
Epoch 8/10
----------
train Loss: 0.9837 Acc: 0.6146
val Loss: 0.9974 Acc: 0.6282
Epoch 9/10
----------
train Loss: 0.9687 Acc: 0.6268
val Loss: 0.9077 Acc: 0.6385
Epoch 10/10
----------
train Loss: 0.9430 Acc: 0.6394
val Loss: 0.9369 Acc: 0.6374
Best val Acc: 0.638492


## Avaliando os modelos quantizados

### W8A8

In [15]:
quant_model=QuantVGG16(8,8)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_vgg16_w8_a8.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantVGG16(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
      (input_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (output_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (weight_quant): WeightQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClampSte()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): StatsFromParameterScaling(
            (parameter_list_stats): _ParameterListStats(
              (first_tracked_param): _ViewParameterWrapper(
                (view_shape_impl): OverOutputChannelView(
                  (permute_impl): Identity()
                )
  

In [16]:
def calculate_metrics(loader, model, device):
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)], output_dict=True)
    kappa = cohen_kappa_score(all_labels, all_preds)
    
    per_class_accuracy = [report[str(i)]['precision'] for i in range(6)]
    mean_accuracy = sum(per_class_accuracy) / len(per_class_accuracy)

    return accuracy, mean_accuracy, kappa, classification_report(all_labels, all_preds, target_names=[str(i) for i in range(6)]), confusion_matrix(all_labels, all_preds)

In [17]:
def plot_confusion_matrix(cm, title, filename):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[str(i) for i in range(6)], yticklabels=[str(i) for i in range(6)])
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(title)
    plt.savefig(filename)
    plt.close()

In [22]:
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w8a8 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w8a8 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w8a8 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_vgg16_w8_a8.png')

Validation Set Metrics for w8a8 quantization:


/tmp/ipykernel_46149/2241107938.py:119: UserWarning: Defining your `__torch_function__` as a plain method is deprecated and will be an error in future, please define it as a classmethod. (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:350.)
  x = torch.flatten(x, 1)


Validation OA: 0.8401
Validation AA: 0.7133
Validation Kappa: 0.7501
Validation Classification Report for w8a8 quantization:
              precision    recall  f1-score   support

           0       0.85      0.98      0.91      1253
           1       0.33      0.02      0.03       126
           2       0.81      0.83      0.82       895
           3       0.49      0.43      0.45        47
           4       0.83      0.69      0.75       182
           5       0.96      0.78      0.86       230

    accuracy                           0.84      2733
   macro avg       0.71      0.62      0.64      2733
weighted avg       0.82      0.84      0.82      2733

Validation Confusion Matrix for w8a8 quantization:
[[1227    1   25    0    0    0]
 [  74    2   49    0    1    0]
 [ 118    3  743   17   10    4]
 [   0    0   21   20    6    0]
 [   4    0   46    4  125    3]
 [  13    0   29    0    9  179]]


In [23]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a8 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w8a8 quantization:")
print(test_report)
print("Test Confusion Matrix for w8a8 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_vgg16_w8_a8.png')

Test Set Metrics for w8a8 quantization:
Test OA: 0.7549
Test AA: 0.6313
Test Kappa: 0.6104
Test Classification Report for w8a8 quantization:
              precision    recall  f1-score   support

           0       0.74      0.97      0.84      1880
           1       0.00      0.00      0.00       189
           2       0.75      0.59      0.66      1344
           3       0.58      0.21      0.31        71
           4       0.90      0.55      0.68       275
           5       0.82      0.91      0.87       346

    accuracy                           0.75      4105
   macro avg       0.63      0.54      0.56      4105
weighted avg       0.72      0.75      0.72      4105

Test Confusion Matrix for w8a8 quantization:
[[1831    0   49    0    0    0]
 [ 127    0   61    0    0    1]
 [ 508    0  788    5    4   39]
 [   2    0   51   15    3    0]
 [   6    0   86    6  150   27]
 [   4    0   17    0   10  315]]


### W8A4

In [25]:
quant_model=QuantVGG16(8,4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_vgg16_w8_a4.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantVGG16(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
      (input_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (output_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (weight_quant): WeightQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClampSte()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): StatsFromParameterScaling(
            (parameter_list_stats): _ParameterListStats(
              (first_tracked_param): _ViewParameterWrapper(
                (view_shape_impl): OverOutputChannelView(
                  (permute_impl): Identity()
                )
  

In [26]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w8a4 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w8a4 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w8a4 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_vgg16_w8_a4.png')

Validation Set Metrics for w8a4 quantization:
Validation OA: 0.8258
Validation AA: 0.6368
Validation Kappa: 0.7271
Validation Classification Report for w8a4 quantization:
              precision    recall  f1-score   support

           0       0.82      0.98      0.90      1253
           1       0.00      0.00      0.00       126
           2       0.82      0.78      0.80       895
           3       0.36      0.43      0.39        47
           4       0.85      0.66      0.75       182
           5       0.96      0.81      0.88       230

    accuracy                           0.83      2733
   macro avg       0.64      0.61      0.62      2733
weighted avg       0.79      0.83      0.80      2733

Validation Confusion Matrix for w8a4 quantization:
[[1228    0   25    0    0    0]
 [  92    0   34    0    0    0]
 [ 154    0  702   32    6    1]
 [   1    0   22   20    4    0]
 [   4    0   47    4  121    6]
 [  11    0   22    0   11  186]]


In [27]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w8a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w8a4 quantization:")
print(test_report)
print("Test Confusion Matrix for w8a4 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_vgg16_w8_a4.png')

Test Set Metrics for w8a4 quantization:
Test OA: 0.7164
Test AA: 0.5908
Test Kappa: 0.5463
Test Classification Report for w8a4 quantization:
              precision    recall  f1-score   support

           0       0.70      0.98      0.81      1880
           1       0.00      0.00      0.00       189
           2       0.74      0.47      0.57      1344
           3       0.47      0.27      0.34        71
           4       0.86      0.46      0.60       275
           5       0.78      0.93      0.85       346

    accuracy                           0.72      4105
   macro avg       0.59      0.52      0.53      4105
weighted avg       0.69      0.72      0.68      4105

Test Confusion Matrix for w8a4 quantization:
[[1845    0   35    0    0    0]
 [ 144    0   44    0    0    1]
 [ 643    0  629   14    5   53]
 [   5    0   42   19    5    0]
 [  12    0   93    7  126   37]
 [   3    0   10    0   11  322]]


### W4A8

In [12]:
quant_model=QuantVGG16(4,8)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_vgg16_w4_a8.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantVGG16(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
      (input_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (output_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (weight_quant): WeightQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClampSte()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): StatsFromParameterScaling(
            (parameter_list_stats): _ParameterListStats(
              (first_tracked_param): _ViewParameterWrapper(
                (view_shape_impl): OverOutputChannelView(
                  (permute_impl): Identity()
                )
  

In [15]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w4a8 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w4a8 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w4a8 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_vgg16_w4_a8.png')

Validation Set Metrics for w4a8 quantization:
Validation OA: 0.8379
Validation AA: 0.6901
Validation Kappa: 0.7482
Validation Classification Report for w4a8 quantization:
              precision    recall  f1-score   support

           0       0.85      0.98      0.91      1253
           1       0.29      0.02      0.03       126
           2       0.83      0.81      0.82       895
           3       0.36      0.53      0.43        47
           4       0.87      0.72      0.79       182
           5       0.95      0.80      0.87       230

    accuracy                           0.84      2733
   macro avg       0.69      0.64      0.64      2733
weighted avg       0.82      0.84      0.82      2733

Validation Confusion Matrix for w4a8 quantization:
[[1223    3   27    0    0    0]
 [  74    2   49    0    1    0]
 [ 118    2  726   40    6    3]
 [   0    0   17   25    5    0]
 [   5    0   34    5  131    7]
 [  20    0   19    0    8  183]]


In [16]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w4a8 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w4a8 quantization:")
print(test_report)
print("Test Confusion Matrix for w4a8 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_vgg16_w4_a8.png')

Test Set Metrics for w4a8 quantization:
Test OA: 0.7452
Test AA: 0.5956
Test Kappa: 0.5961
Test Classification Report for w4a8 quantization:
              precision    recall  f1-score   support

           0       0.73      0.98      0.84      1880
           1       0.00      0.00      0.00       189
           2       0.76      0.55      0.64      1344
           3       0.42      0.31      0.35        71
           4       0.87      0.54      0.67       275
           5       0.79      0.91      0.85       346

    accuracy                           0.75      4105
   macro avg       0.60      0.55      0.56      4105
weighted avg       0.72      0.75      0.71      4105

Test Confusion Matrix for w4a8 quantization:
[[1838    0   42    0    0    0]
 [ 132    0   57    0    0    0]
 [ 531    1  735   21    6   50]
 [   3    0   41   22    4    1]
 [  10    0   75   10  149   31]
 [   8    0   11    0   12  315]]


### W4A4

In [31]:
quant_model=QuantVGG16(4,4)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
quant_model.load_state_dict(torch.load("qat_vgg16_w4_a4.pth", map_location=device))
quant_model = quant_model.to(device)
quant_model.eval()

QuantVGG16(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)
      (input_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (output_quant): ActQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
      )
      (weight_quant): WeightQuantProxyFromInjector(
        (_zero_hw_sentinel): StatelessBuffer()
        (tensor_quant): RescalingIntQuant(
          (int_quant): IntQuant(
            (float_to_int_impl): RoundSte()
            (tensor_clamp_impl): TensorClampSte()
            (delay_wrapper): DelayWrapper(
              (delay_impl): _NoDelay()
            )
          )
          (scaling_impl): StatsFromParameterScaling(
            (parameter_list_stats): _ParameterListStats(
              (first_tracked_param): _ViewParameterWrapper(
                (view_shape_impl): OverOutputChannelView(
                  (permute_impl): Identity()
                )
  

In [32]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Validation Set Metrics for w4a4 quantization:")
valid_accuracy, valid_mean_accuracy, valid_kappa, valid_report, valid_conf_matrix = calculate_metrics(valid_loader, quant_model, device)
print(f"Validation OA: {valid_accuracy:.4f}")
print(f"Validation AA: {valid_mean_accuracy:.4f}")
print(f"Validation Kappa: {valid_kappa:.4f}")
print("Validation Classification Report for w4a4 quantization:")
print(valid_report)
print("Validation Confusion Matrix for w4a4 quantization:")
print(valid_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de validação
plot_confusion_matrix(valid_conf_matrix, 'Validation Confusion Matrix', 'validation_confusion_matrix_vgg16_w4_a4.png')

Validation Set Metrics for w4a4 quantization:
Validation OA: 0.8119
Validation AA: 0.6499
Validation Kappa: 0.7071
Validation Classification Report for w4a4 quantization:
              precision    recall  f1-score   support

           0       0.86      0.96      0.91      1253
           1       0.15      0.02      0.03       126
           2       0.77      0.81      0.79       895
           3       0.24      0.47      0.32        47
           4       0.91      0.52      0.66       182
           5       0.97      0.73      0.83       230

    accuracy                           0.81      2733
   macro avg       0.65      0.58      0.59      2733
weighted avg       0.80      0.81      0.80      2733

Validation Confusion Matrix for w4a4 quantization:
[[1208    7   38    0    0    0]
 [  79    2   43    0    1    1]
 [ 109    4  725   56    1    0]
 [   0    0   22   22    3    0]
 [   5    0   65   13   94    5]
 [   5    0   53    0    4  168]]


In [33]:
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="UndefinedMetricWarning")
print("Test Set Metrics for w4a4 quantization:")
test_accuracy, test_mean_accuracy, test_kappa, test_report, test_conf_matrix = calculate_metrics(test_loader, quant_model, device)
print(f"Test OA: {test_accuracy:.4f}")
print(f"Test AA: {test_mean_accuracy:.4f}")
print(f"Test Kappa: {test_kappa:.4f}")
print("Test Classification Report for w4a4 quantization:")
print(test_report)
print("Test Confusion Matrix for w4a4 quantization:")
print(test_conf_matrix)

# Plotando e salvando a matriz de confusão para o conjunto de teste
plot_confusion_matrix(test_conf_matrix, 'Test Confusion Matrix', 'test_confusion_matrix_vgg16_w4_a4.png')

Test Set Metrics for w4a4 quantization:
Test OA: 0.7306
Test AA: 0.5846
Test Kappa: 0.5705
Test Classification Report for w4a4 quantization:
              precision    recall  f1-score   support

           0       0.74      0.97      0.84      1880
           1       0.00      0.00      0.00       189
           2       0.70      0.56      0.62      1344
           3       0.37      0.28      0.32        71
           4       0.92      0.32      0.47       275
           5       0.78      0.90      0.84       346

    accuracy                           0.73      4105
   macro avg       0.58      0.51      0.52      4105
weighted avg       0.70      0.73      0.70      4105

Test Confusion Matrix for w4a4 quantization:
[[1828    0   52    0    0    0]
 [ 128    0   59    0    0    2]
 [ 520    1  751   24    0   48]
 [   3    0   47   20    1    0]
 [   6    0  136   10   87   36]
 [   1    0   25    0    7  313]]


# Exportando os modelos para ONNX -> FINN

In [6]:
cd Classif-RetinopatiaDiabetica/

/home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica


In [18]:
qat_models = [(8,8,"qat_vgg16_w8_a8.pth","/classifier-vgg16-w8a8-ready.onnx"),(8,4,"qat_vgg16_w8_a4.pth","/classifier-vgg16-w8a4-ready.onnx"),
             (4,8,"qat_vgg16_w4_a8.pth","/classifier-vgg16-w4a8-ready.onnx"),(4,4,"qat_vgg16_w4_a4.pth","/classifier-vgg16-w4a4-ready.onnx"),
             (8,2,"qat_vgg16_w8_a2.pth","/classifier-vgg16-w8a2-ready.onnx"),(2,8,"qat_vgg16_w2_a8.pth","/classifier-vgg16-w2a8-ready.onnx")]
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"

for w,a,model_path,onnx_path in qat_models:
    print(f"Exportando modelo W{w}A{a}")
    quant_model = QuantVGG16(w,a)
    quant_model.load_state_dict(torch.load(model_path, map_location=device))
    quant_model.cpu()
    quant_model.eval()
    
    quant_model_export = quant_model
    ready_model_filename = model_dir + onnx_path

    # model -> onnx
    input_shape = (1,3,224,224)
    input_a = np.random.randn(*input_shape).astype(np.float32)
    input_t = torch.from_numpy(input_a)
    export_qonnx(quant_model_export,export_path=ready_model_filename,input_t=input_t) 
    qonnx_cleanup(ready_model_filename,out_file=ready_model_filename)
    # onnx -> finn
    wrapped_model = ModelWrapper(ready_model_filename)
    wrapped_model.set_tensor_datatype(wrapped_model.graph.input[0].name,DataType["FLOAT32"])
    wrapped_model = wrapped_model.transform(ConvertQONNXtoFINN())
    wrapped_model.save(ready_model_filename)
    print(f"Model saved to {ready_model_filename}")

Exportando modelo W8A8
Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w8a8-ready.onnx
Exportando modelo W8A4
Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w8a4-ready.onnx
Exportando modelo W4A8
Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w4a8-ready.onnx
Exportando modelo W4A4
Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w4a4-ready.onnx
Exportando modelo W8A2
Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w8a2-ready.onnx
Exportando modelo W2A8
Model saved to /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w2a8-ready.onnx


# Gerando Resultados estimados

In [19]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_VGG16-w8a8"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w8a8 = build_cfg.DataflowBuildConfig(
    steps=[
        "step_tidy_up",
        "step_streamline",
        "step_convert_to_hw",
        "step_create_dataflow_partition",
        "step_specialize_layers",
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [20]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_VGG16-w8a4"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w8a4 = build_cfg.DataflowBuildConfig(
    steps=[
        "step_tidy_up",
        "step_streamline",
        "step_convert_to_hw",
        "step_create_dataflow_partition",
        "step_specialize_layers",
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [22]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_VGG16-w4a8"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w4a8 = build_cfg.DataflowBuildConfig(
    steps=[
        "step_tidy_up",
        "step_streamline",
        "step_convert_to_hw",
        "step_create_dataflow_partition",
        "step_specialize_layers",
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [22]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_VGG16-w4a4"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w4a4 = build_cfg.DataflowBuildConfig(
    steps=[
        "step_tidy_up",
        "step_streamline",
        "step_convert_to_hw",
        "step_create_dataflow_partition",
        "step_specialize_layers",
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [19]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_VGG16-w8a2"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w8a2 = build_cfg.DataflowBuildConfig(
    steps=[
        "step_tidy_up",
        "step_streamline",
        "step_convert_to_hw",
        "step_create_dataflow_partition",
        "step_specialize_layers",
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [20]:
model_dir = os.environ['FINN_ROOT'] + "/notebooks/Classif-RetinopatiaDiabetica"
output_dir=model_dir + "/generated_outputs_VGG16-w2a8"
#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")
    
estimate_w2a8 = build_cfg.DataflowBuildConfig(
    steps=[
        "step_tidy_up",
        "step_streamline",
        "step_convert_to_hw",
        "step_create_dataflow_partition",
        "step_specialize_layers",
        "step_target_fps_parallelization",
        "step_apply_folding_config",
        "step_minimize_bit_width",
        "step_generate_estimate_reports",
],
    target_fps=10,
    output_dir=output_dir,
    synth_clk_period_ns=10.0,
    fpga_part="XCU250",
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
    ],
)

In [23]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-vgg16-w8a8-ready.onnx", estimate_w8a8)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w8a8-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w8a8
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w8a8/build_dataflow.log
Running step: step_tidy_up [1/9]
Running step: step_streamline [2/9]
Running step: step_convert_to_hw [3/9]
Running step: step_create_dataflow_partition [4/9]
Running step: step_specialize_layers [5/9]
Running step: step_target_fps_parallelization [6/9]
Running step: step_apply_folding_config [7/9]
Running step: step_minimize_bit_width [8/9]
Running step: step_generate_estimate_reports [9/9]
Completed successfully
CPU times: user 15.3 s, sys: 6.21 s, total: 21.5 s
Wall time: 21.5 s


0

In [24]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-vgg16-w8a4-ready.onnx", estimate_w8a4)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w8a4-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w8a4
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w8a4/build_dataflow.log
Running step: step_tidy_up [1/9]
Running step: step_streamline [2/9]
Running step: step_convert_to_hw [3/9]
Running step: step_create_dataflow_partition [4/9]
Running step: step_specialize_layers [5/9]
Running step: step_target_fps_parallelization [6/9]
Running step: step_apply_folding_config [7/9]
Running step: step_minimize_bit_width [8/9]
Running step: step_generate_estimate_reports [9/9]
Completed successfully
CPU times: user 13.4 s, sys: 6.14 s, total: 19.5 s
Wall time: 19.7 s


0

In [23]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-vgg16-w4a8-ready.onnx", estimate_w4a8)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w4a8-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w4a8
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w4a8/build_dataflow.log
Running step: step_tidy_up [1/9]
Running step: step_streamline [2/9]
Running step: step_convert_to_hw [3/9]
Running step: step_create_dataflow_partition [4/9]
Running step: step_specialize_layers [5/9]
Running step: step_target_fps_parallelization [6/9]
Running step: step_apply_folding_config [7/9]
Running step: step_minimize_bit_width [8/9]
Running step: step_generate_estimate_reports [9/9]
Completed successfully
CPU times: user 8.4 s, sys: 6.71 s, total: 15.1 s
Wall time: 16.4 s


0

In [25]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-vgg16-w4a4-ready.onnx", estimate_w4a4)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w4a4-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w4a4
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w4a4/build_dataflow.log
Running step: step_tidy_up [1/9]
Running step: step_streamline [2/9]
Running step: step_convert_to_hw [3/9]
Running step: step_create_dataflow_partition [4/9]
Running step: step_specialize_layers [5/9]
Running step: step_target_fps_parallelization [6/9]
Running step: step_apply_folding_config [7/9]
Running step: step_minimize_bit_width [8/9]
Running step: step_generate_estimate_reports [9/9]
Completed successfully
CPU times: user 13 s, sys: 6.13 s, total: 19.2 s
Wall time: 19.2 s


0

In [24]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-vgg16-w8a2-ready.onnx", estimate_w8a2)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w8a2-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w8a2
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w8a2/build_dataflow.log
Running step: step_tidy_up [1/9]
Running step: step_streamline [2/9]
Running step: step_convert_to_hw [3/9]
Running step: step_create_dataflow_partition [4/9]
Running step: step_specialize_layers [5/9]
Running step: step_target_fps_parallelization [6/9]
Running step: step_apply_folding_config [7/9]
Running step: step_minimize_bit_width [8/9]
Running step: step_generate_estimate_reports [9/9]
Completed successfully
CPU times: user 7.25 s, sys: 6.39 s, total: 13.6 s
Wall time: 15 s


0

In [25]:
%%time
build.build_dataflow_cfg(model_dir + "/classifier-vgg16-w2a8-ready.onnx", estimate_w2a8)

Building dataflow accelerator from /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/classifier-vgg16-w2a8-ready.onnx
Intermediate outputs will be generated in /tmp/finn_dev_isadora
Final outputs will be generated in /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w2a8
Build log is at /home/isadora/Xilinx/finn/notebooks/Classif-RetinopatiaDiabetica/generated_outputs_VGG16-w2a8/build_dataflow.log
Running step: step_tidy_up [1/9]
Running step: step_streamline [2/9]
Running step: step_convert_to_hw [3/9]
Running step: step_create_dataflow_partition [4/9]
Running step: step_specialize_layers [5/9]
Running step: step_target_fps_parallelization [6/9]
Running step: step_apply_folding_config [7/9]
Running step: step_minimize_bit_width [8/9]
Running step: step_generate_estimate_reports [9/9]
Completed successfully
CPU times: user 8.28 s, sys: 6.71 s, total: 15 s
Wall time: 15.9 s


0